# 04 — Evaluate Supervisor Agent

This notebook evaluates the **Supply Chain Supervisor Agent** using MLflow Agent Evaluation.

We benchmark:
* **Latency** — end-to-end response time (mean, median, P95) across question categories
* **Tool-Call Correctness** — did the supervisor route to the correct Genie space?
* **Tool-Call Efficiency** — no redundant calls to irrelevant sub-agents?
* **Response Quality** — accurate, specific, actionable answers?

**Prerequisites:** Run notebooks 01–03 first to create data, Genie spaces, and the supervisor agent.

In [0]:
%pip install databricks-sdk openai "mlflow>=3.12.0" pandas --upgrade --quiet
dbutils.library.restartPython()

In [0]:
"""Setup: imports, configuration, MLflow experiment, and OpenAI client for the supervisor endpoint."""
import os
import time
import json
import pandas as pd
import mlflow
from openai import OpenAI
from databricks.sdk import WorkspaceClient

# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CONFIGURATION — Only change these if you need a different setup           │
# └──────────────────────────────────────────────────────────────────────────┘

# LLM model for the MLflow judges. Any Foundation Model API endpoint works.
# Format: "databricks:/<endpoint-name>"
JUDGE_MODEL = "databricks:/databricks-claude-sonnet-4"

# ┌──────────────────────────────────────────────────────────────────────────┐
# │ AUTO-DISCOVERED — No changes needed below this line                      │
# └──────────────────────────────────────────────────────────────────────────┘

w = WorkspaceClient()

# Current user (for experiment path and endpoint filtering)
_username = (
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
)

# Discover the supervisor endpoint: filter mas-* endpoints created by current user.
# AgentBricks naming convention: mas-<first-8-chars-of-agent-id>-endpoint
_my_endpoints = [
    ep.name
    for ep in w.serving_endpoints.list()
    if ep.name.startswith("mas-") and ep.name.endswith("-endpoint") and ep.creator == _username
]

if len(_my_endpoints) == 1:
    ENDPOINT_NAME = _my_endpoints[0]
elif len(_my_endpoints) > 1:
    # Multiple supervisor agents owned by this user — pick the most recently created
    # or override manually: ENDPOINT_NAME = "mas-XXXXXXXX-endpoint"
    print(f"Found {len(_my_endpoints)} supervisor endpoints: {_my_endpoints}")
    print("Using the first one. Override ENDPOINT_NAME above if needed.")
    ENDPOINT_NAME = _my_endpoints[0]
else:
    raise ValueError(
        f"No mas-*-endpoint found for user '{_username}'. "
        f"Run notebook 03 first to create the supervisor agent."
    )

# Derive the short agent ID from the endpoint name
AGENT_ID = ENDPOINT_NAME.replace("mas-", "").replace("-endpoint", "")

# Host and token (from current notebook session)
HOST = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
TOKEN = (
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
)

# MLflow experiment — scoped to the current user's home directory
EXPERIMENT_NAME = f"/Users/{_username}/supply-chain-supervisor-eval"
mlflow.set_experiment(EXPERIMENT_NAME)

# OpenAI-compatible client for the supervisor endpoint
client = OpenAI(api_key=TOKEN, base_url=f"{HOST}/serving-endpoints")

print(f"Endpoint:    {ENDPOINT_NAME}")
print(f"Agent ID:    {AGENT_ID}")
print(f"Experiment:  {EXPERIMENT_NAME}")
print(f"Judge Model: {JUDGE_MODEL}")

## Evaluation Dataset

We define 12 evaluation questions spanning three categories:
* **Procurement** (5) — routed to `procurement_inventory` Genie space
* **Logistics** (5) — routed to `logistics_fulfillment` Genie space
* **Cross-domain** (2) — requires both tools with synthesis

Each question has an expected tool routing and description of what a correct answer looks like.

In [0]:
"""Define the evaluation dataset with questions, expected routing, and descriptions."""

eval_cases = [
    # Procurement domain (2)
    {
        "question": "Which supplier has the highest total spend?",
        "expected_tool": "procurement_inventory",
        "category": "procurement",
        "description": "Should query procurement_metrics for total spend by supplier",
    },
    {
        "question": "Are any materials below their reorder point at warehouse WH-EAST?",
        "expected_tool": "procurement_inventory",
        "category": "procurement",
        "description": "Should check inventory table for items below reorder_point",
    },
    # Logistics domain (2)
    {
        "question": "What is the on-time delivery rate for air freight?",
        "expected_tool": "logistics_fulfillment",
        "category": "logistics",
        "description": "Should query logistics_metrics filtering transport_mode=Air",
    },
    {
        "question": "What is the longest route by distance?",
        "expected_tool": "logistics_fulfillment",
        "category": "logistics",
        "description": "Should query routes table ordered by distance_km",
    },
    # Cross-domain (1)
    {
        "question": "Compare on-time rates between suppliers and carriers",
        "expected_tool": "both",
        "category": "cross-domain",
        "description": "Should get on-time from procurement_metrics and logistics_metrics",
    },
]

eval_df = pd.DataFrame(eval_cases)
print(f"Evaluation dataset: {len(eval_df)} questions")
print(f"  Procurement:  {len(eval_df[eval_df['category'] == 'procurement'])}")
print(f"  Logistics:    {len(eval_df[eval_df['category'] == 'logistics'])}")
print(f"  Cross-domain: {len(eval_df[eval_df['category'] == 'cross-domain'])}")
display(eval_df)

## Invoke & Evaluate

The `predict_fn` pattern invokes the agent live during `mlflow.genai.evaluate()`, so each trace captures **real execution time** and all scorers produce assessments on the same trace.

**Scorers:**
| # | Name | Type | What it checks |
|---|------|------|----------------|
| 1 | `routing_correctness` | `@scorer` (deterministic) | Correct Genie space selected |
| 2 | `request_latency_seconds` | `@scorer` (numeric) | Wall-clock latency with p90 |
| 3 | `tool_call_efficiency` | `make_judge` (LLM) | No redundant tool calls |
| 4 | `response_quality` | `make_judge` (LLM) | Relevant, specific, complete |

In [0]:
"""Helper function to invoke the supervisor agent and capture response + latency + tool calls."""


def invoke_supervisor(question: str) -> dict:
    """Invoke the supervisor agent and capture response, latency, and tool call metadata.

    Args:
        question: The natural language question to send to the supervisor.

    Returns:
        Dict with response text, latency_seconds, response_id, and tools_called list.
    """
    start_time = time.time()

    response = client.responses.create(
        model=ENDPOINT_NAME,
        input=[{"role": "user", "content": question}],
    )

    latency = time.time() - start_time

    # Extract response text and tool calls from output items
    response_text = ""
    tools_called = []
    for item in response.output:
        if hasattr(item, "content"):
            for content_block in item.content:
                if hasattr(content_block, "text"):
                    response_text += content_block.text
        if hasattr(item, "type") and item.type == "function_call":
            tools_called.append(item.name)

    return {
        "response": response_text,
        "latency_seconds": latency,
        "response_id": response.id,
        "tools_called": tools_called,
    }


print("✓ invoke_supervisor defined")

## Define Scorers & Judges

### Design decisions
* **Built-in `ToolCallCorrectness` won't work** — the managed supervisor executes tools server-side; autolog only captures a single `CHAT_MODEL` span with no `TOOL` spans.
* **`@scorer` for deterministic checks** — routing + latency are code-based (no LLM cost, instant, 100% reproducible).
* **`make_judge` for qualitative assessment** — efficiency + quality need LLM reasoning with clear rubrics.

In [0]:
"""Define all scorers: deterministic @scorer + make_judge (passed directly as scorers)."""

from typing import Literal
from mlflow.genai.scorers import scorer
from mlflow.genai.judges import make_judge
from mlflow.entities import Feedback

JUDGE_MODEL = "databricks:/databricks-claude-sonnet-4"

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# SCORER 1: Routing Correctness (deterministic — no LLM cost)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


@scorer
def routing_correctness(outputs, expectations):
    """Did the supervisor route to the correct Genie space tool?

    Rubric:
      PASS: expected_tool appears in the list of tools actually called.
            For cross-domain (expected_tool="both"), both tools must be called.
      FAIL: Expected tool was not called, or for cross-domain, one/both missing.
    """
    tools_called = outputs.get("tools_called", [])
    expected = expectations.get("expected_tool", "")

    if expected == "both":
        called_set = set(tools_called)
        required = {"procurement_inventory", "logistics_fulfillment"}
        missing = required - called_set
        if not missing:
            return Feedback(value="yes", rationale=f"Both tools called: {tools_called}")
        return Feedback(value="no", rationale=f"Missing {missing}. Called: {tools_called}")
    else:
        if expected in tools_called:
            return Feedback(value="yes", rationale=f"Correctly routed to '{expected}'")
        return Feedback(
            value="no", rationale=f"Expected '{expected}', called: {tools_called}"
        )


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# SCORER 2: Request Latency (numeric with aggregations)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


@scorer(aggregations=["mean", "median", "min", "max", "p90"])
def request_latency_seconds(outputs) -> float:
    """Wall-clock latency per request in seconds.

    Aggregated as mean, median, min, max, p90 in the MLflow experiment.
    """
    return outputs.get("latency_seconds", 0.0)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# SCORER 3: Tool-Call Efficiency (LLM judge — passed directly as scorer)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#
# make_judge returns an object that IS a scorer — no @scorer wrapper needed.
# {{ inputs }} and {{ outputs }} receive the full dicts from predict_fn.
# The rubric explicitly directs the judge to evaluate the "response" field.
#
tool_call_efficiency = make_judge(
    name="tool_call_efficiency",
    instructions="""You are evaluating whether a multi-agent supply chain system used its tools efficiently.

Context:
- This system has TWO Genie space tools: "procurement_inventory" (suppliers, materials, POs, stock) and "logistics_fulfillment" (carriers, shipments, routes, deliveries).
- The supervisor routes questions to one or both tools.

User Question: {{ inputs }}
Agent Response (evaluate only the "response" field below): {{ outputs }}

Rubric — return "yes" if ALL criteria are met, "no" if ANY are violated:

1. MINIMAL TOOL USE: A single-domain question should only produce data from one domain.
   - PASS: "Which supplier has the highest spend?" → response discusses only suppliers/procurement.
   - FAIL: "Which supplier has the highest spend?" → response also discusses carrier shipping rates.

2. COMPLETE TOOL USE: A cross-domain question (comparing/combining procurement AND logistics) must include data from both domains.
   - PASS: "Compare supplier vs carrier on-time rates" → includes both supplier and carrier metrics.
   - FAIL: "Compare supplier vs carrier on-time rates" → only discusses suppliers.

3. NO REDUNDANCY: The response should not repeat the same data points multiple times.
   - PASS: Lists each data point once with a clear summary.
   - FAIL: Repeats the same table data in prose and again in a summary.""",
    feedback_value_type=Literal["yes", "no"],
    model=JUDGE_MODEL,
)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# SCORER 4: Response Quality (LLM judge — 4-criteria rubric)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

response_quality = make_judge(
    name="response_quality",
    instructions="""You are evaluating the answer quality of a Supply Chain AI agent.

User Question: {{ inputs }}
Agent Response (evaluate only the "response" field below): {{ outputs }}

Score on 4 criteria. Return "yes" if ≥3 are satisfied, "no" otherwise.

1. RELEVANCE — Does the response directly address what was asked?
   - PASS: Question about lead times → response provides lead time data.
   - FAIL: Response discusses unrelated topics or gives a generic non-answer.

2. SPECIFICITY — Does it include concrete data (numbers, names, percentages)?
   - PASS: "Apex Steel Corp has the highest spend at $47,200" or "Average lead time: 21 days."
   - FAIL: "There are several suppliers with varying spend levels" (vague, no data).

3. COMPLETENESS — Does it fully answer without requiring follow-up?
   - PASS: "3 materials below reorder point: Cold Rolled Steel (82/150), ..." (lists all).
   - FAIL: "Some materials are below reorder point" (no names or counts).

4. CLARITY — Is the information well-organized and easy to act on?
   - PASS: Structured format (tables, bullets, headers) for multi-item answers.
   - FAIL: Wall of unformatted text mixing topics without separation.""",
    feedback_value_type=Literal["yes", "no"],
    model=JUDGE_MODEL,
)


print("✓ Scorers defined:")
print("  1. routing_correctness   — @scorer, deterministic")
print("  2. request_latency_seconds — @scorer, numeric (mean/median/p90)")
print(f"  3. tool_call_efficiency  — make_judge, passed directly ({JUDGE_MODEL})")
print(f"  4. response_quality      — make_judge, passed directly ({JUDGE_MODEL})")

In [0]:
"""Run unified evaluation: predict_fn invokes the agent with inter-question delay."""

import mlflow

# Enable autolog so the internal OpenAI call appears as a span in each trace
mlflow.openai.autolog()

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# predict_fn: invokes supervisor with delay to avoid Genie API rate limits
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

QUESTION_DELAY_SECONDS = 3  # Avoids Genie API rate limit (60s window)


def predict_fn(question: str) -> dict:
    """Invoke the supervisor and return structured outputs for scoring.

    Includes a delay before each call to stay within Genie API rate limits.
    With 5 questions at ~15s each + 3s delay, total ~90s spread avoids throttling.
    """
    time.sleep(QUESTION_DELAY_SECONDS)
    result = invoke_supervisor(question)
    return {
        "response": result["response"],
        "tools_called": result["tools_called"],
        "latency_seconds": result["latency_seconds"],
    }


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Build evaluation dataset with expectations
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
eval_data = []
for _, row in eval_df.iterrows():
    eval_data.append(
        {
            "inputs": {"question": row["question"]},
            "expectations": {
                "expected_tool": row["expected_tool"],
                "category": row["category"],
            },
        }
    )

eval_dataset = pd.DataFrame(eval_data)
print(f"Evaluation dataset: {len(eval_dataset)} questions")
print(f"Question delay: {QUESTION_DELAY_SECONDS}s")
print("=" * 60)
print("Running evaluation (predict_fn + 4 scorers)...")
print()

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Run evaluation
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
with mlflow.start_run(run_name="supervisor_eval_v7") as run:
    mlflow.log_param("endpoint_name", ENDPOINT_NAME)
    mlflow.log_param("agent_id", AGENT_ID)
    mlflow.log_param("num_questions", len(eval_data))
    mlflow.log_param("judge_model", JUDGE_MODEL)
    mlflow.log_param("question_delay_s", QUESTION_DELAY_SECONDS)

    eval_result = mlflow.genai.evaluate(
        data=eval_dataset,
        predict_fn=predict_fn,
        scorers=[
            routing_correctness,
            request_latency_seconds,
            tool_call_efficiency,
            response_quality,
        ],
    )

    print(f"\n{'=' * 60}")
    print(f"MLflow Run: {run.info.run_id}")
    print(f"\nMetrics:")
    for metric_name, metric_value in sorted(eval_result.metrics.items()):
        print(f"  {metric_name}: {metric_value}")

mlflow.openai.autolog(disable=True)
print("\n✓ Evaluation complete — all assessments on real traces.")

## Evaluation Results Summary

Combined view of latency benchmarks and judge scores with actionable recommendations.

In [0]:
"""Display comprehensive evaluation summary from eval_result: metrics, per-trace assessments, and recommendations."""

import mlflow

print("=" * 60)
print("SUPERVISOR AGENT EVALUATION SUMMARY")
print("=" * 60)

# ━━ Aggregate Metrics from eval_result ━━
print("\n\U0001f4ca AGGREGATE METRICS")
for metric_name, metric_value in sorted(eval_result.metrics.items()):
    print(f"  {metric_name}: {metric_value}")

# ━━ Per-Trace Assessment Details ━━
print(f"\n\U0001f9d1\u200d\u2696\ufe0f PER-TRACE ASSESSMENTS")
client_mlflow = mlflow.tracking.MlflowClient()
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
traces = client_mlflow.search_traces(
    experiment_ids=[experiment.experiment_id],
    max_results=12,
    order_by=["timestamp_ms DESC"],
)

# Collect assessment stats
assessment_stats = {}
latencies = []
for t in traces:
    for a in t.info.assessments:
        if a.name not in assessment_stats:
            assessment_stats[a.name] = {"yes": 0, "no": 0, "total": 0, "values": []}
        assessment_stats[a.name]["total"] += 1
        if a.feedback and a.feedback.value == "yes":
            assessment_stats[a.name]["yes"] += 1
        elif a.feedback and a.feedback.value == "no":
            assessment_stats[a.name]["no"] += 1
        if a.name == "request_latency_seconds" and a.feedback:
            try:
                latencies.append(float(a.feedback.value))
            except (ValueError, TypeError):
                pass

# Display categorical scorers
for scorer_name in ["routing_correctness", "tool_call_efficiency", "response_quality"]:
    if scorer_name in assessment_stats:
        stats = assessment_stats[scorer_name]
        pct = 100 * stats["yes"] / stats["total"] if stats["total"] > 0 else 0
        emoji = "\u2713" if pct >= 90 else "\u26a1" if pct >= 70 else "\u26a0\ufe0f"
        print(f"  {emoji} {scorer_name}: {stats['yes']}/{stats['total']} ({pct:.0f}%)")

# Display latency from assessments
if latencies:
    import statistics
    print(f"\n\u23f1\ufe0f  LATENCY (from trace assessments)")
    print(f"  Mean:   {statistics.mean(latencies):.2f}s")
    print(f"  Median: {statistics.median(latencies):.2f}s")
    sorted_lat = sorted(latencies)
    p90_idx = int(0.9 * len(sorted_lat))
    print(f"  P90:    {sorted_lat[p90_idx] if sorted_lat else 0:.2f}s")
    print(f"  Min:    {min(latencies):.2f}s")
    print(f"  Max:    {max(latencies):.2f}s")

# ━━ Recommendations ━━
print(f"\n\U0001f4a1 RECOMMENDATIONS")
mean_lat = statistics.mean(latencies) if latencies else 0
if mean_lat > 30:
    print("  \u26a0\ufe0f  High latency. Genie spaces add SQL generation + warehouse overhead.")
    print("     - Expected for managed supervisor \u2192 Genie space chains")
    print("     - Cross-domain questions call both tools sequentially")
elif mean_lat > 15:
    print("  \u26a1 Moderate latency. Expected for multi-hop orchestration.")
else:
    print("  \u2713 Latency within acceptable range.")

routing_stats = assessment_stats.get("routing_correctness", {})
if routing_stats.get("no", 0) > 0:
    print(f"\n  \u26a0\ufe0f  Routing issues detected ({routing_stats['no']} misrouted):")
    print("     Fix: Add guidelines in the supervisor's Examples tab")
else:
    print("\n  \u2713 Perfect routing \u2014 all questions sent to the correct Genie space.")

## Next Steps

* **Improve routing**: Go to supervisor configuration → Examples tab → add guidelines based on misrouted questions
* **Add human review**: Share MLflow experiment with SMEs to rate responses in the Review App
* **Monitor production**: Set up Lakehouse Monitoring on the inference table for ongoing drift detection
* **Iterate on judges**: Refine judge instructions based on false positives/negatives observed above
* **Expand dataset**: Add edge cases (ambiguous questions, out-of-scope queries) to stress-test routing